# Pipeline Test Notebook

This notebook tests each step of the pipeline using a locally run model.

## prerequisites
Ensure you have `llama-cpp-python` installed and a GGUF model available in the `models/` directory as specified in `config/models.yaml`.

In [ ]:
import os
import sys
import yaml

# Add project root to path
sys.path.append(os.path.abspath('.'))

from kg_constructors.pipeline import ConstructionPipeline
from config.config_loader import load_model_config

## 1. Load Configuration
Check if the local model is correctly defined.

In [ ]:
models = load_model_config()
print("Loaded models configuration:")
print(yaml.dump(models.get('local', {}))) # Show only local models

# Set the model to test
TEST_MODEL = 'llama31_8b_instruct'  # Ensure this matches a name in your config
TEST_CONCEPT = 'cat'
TEST_PROPERTY = 'size'
TEST_TEMPLATE = 'emotions' # Using emotions as a placeholder template

## 2. Initialize Pipeline
We will attempt to initialize the pipeline. This might fail if the model file is missing.

In [ ]:
try:
    pipeline = ConstructionPipeline(
        concept=TEST_CONCEPT,
        prompt_template_name=TEST_TEMPLATE,
        property=TEST_PROPERTY,
        model_name=TEST_MODEL,
        ontology_path='test_output.jsonl',
        n_ctx=2048 # Kwarg for LocalClient/Llama
    )
    print("Pipeline initialized successfully.")
except ValueError as e:
    print(f"Initialization failed: {e}")
except Exception as e:
    print(f"An error occurred (check if model path exists): {e}")

## 3. Run Pipeline Step
Execute the pipeline. This triggers model loading and generation.

In [ ]:
if 'pipeline' in locals():
    try:
        print(f"Running pipeline for {TEST_CONCEPT}...")
        pipeline.run()
        print("Pipeline run finished.")
    except Exception as e:
        print(f"Execution failed: {e}")
else:
    print("Pipeline not initialized, skipping run.")

## 4. Verify Output
Check if the output file was created and contains valid JSONL.

In [ ]:
import json

output_file = 'test_output.jsonl'
if os.path.exists(output_file):
    print(f"Reading {output_file}...")
    with open(output_file, 'r') as f:
        for line in f:
            data = json.loads(line)
            print(json.dumps(data, indent=2))
else:
    print(f"Output file {output_file} not found.")

## 5. Test CLI
Verify the command line interface works as expected.

In [ ]:
!python main.py --help

In [ ]:
# dry-run the CLI if possible, but running a full model might take time.
# We'll just verify it starts up.
import subprocess

cmd = [
    "python", "main.py",
    "--concept", "dog",
    "--model", TEST_MODEL,
    "--property", "size",
    "--ontology", "test_cli_output.jsonl"
]

print(f"Testing command: {' '.join(cmd)}")
# Uncomment the following lines to actually run it (might take time/memory)
# result = subprocess.run(cmd, capture_output=True, text=True)
# print(result.stdout)
# print(result.stderr)